In [1]:
from pathlib import Path
import os
import sys
import subprocess

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "quran_asr").exists() or not (PROJECT_ROOT / "configs").exists():
    raise RuntimeError(
        f"Notebook harus dijalankan dari root project.\n"
        f"Posisi sekarang: {PROJECT_ROOT}"
    )

CONFIG_PATH = PROJECT_ROOT / "configs" / "local_3050.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config belum ada: {CONFIG_PATH}")

CACHE_DIR = PROJECT_ROOT / ".cache" / "hf"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(CACHE_DIR / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR / "transformers")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)
print("HF_HOME:", os.environ["HF_HOME"])
print("Python:", sys.executable)

PROJECT_ROOT: /home/jrilym/Projects/AI/model-asr-quran
CONFIG_PATH: /home/jrilym/Projects/AI/model-asr-quran/configs/local_3050.yaml
HF_HOME: /home/jrilym/Projects/AI/model-asr-quran/.cache/hf
Python: /home/jrilym/Projects/AI/model-asr-quran/.venv/bin/python


In [2]:
def run_cmd(args):
    print("Running:")
    print(" ".join(str(arg) for arg in args))
    subprocess.run(args, check=True, cwd=PROJECT_ROOT)

In [3]:
RAW_AUDIO_DIR = PROJECT_ROOT / "data" / "raw" / "audio"
RAW_TEXT_FILE = PROJECT_ROOT / "data" / "raw" / "text" / "quran_uthmani.json"

if not RAW_AUDIO_DIR.exists() or not any(RAW_AUDIO_DIR.iterdir()):
    raise RuntimeError(f"Audio belum ada di: {RAW_AUDIO_DIR}")

if not RAW_TEXT_FILE.exists():
    raise FileNotFoundError(f"Text Quran belum ada: {RAW_TEXT_FILE}")

audio_files = list(RAW_AUDIO_DIR.rglob("*.*"))

print("Raw audio OK:", RAW_AUDIO_DIR)
print("Jumlah audio:", len(audio_files))
print("Quran text OK:", RAW_TEXT_FILE)

print("\nContoh audio:")
for file in audio_files[:5]:
    print("-", file.relative_to(PROJECT_ROOT))

Raw audio OK: /home/jrilym/Projects/AI/model-asr-quran/data/raw/audio
Jumlah audio: 24946
Quran text OK: /home/jrilym/Projects/AI/model-asr-quran/data/raw/text/quran_uthmani.json

Contoh audio:
- data/raw/audio/Husary_128kbps_Mujawwad/001003.mp3
- data/raw/audio/Husary_128kbps_Mujawwad/001003.md5
- data/raw/audio/Husary_128kbps_Mujawwad/001002.mp3
- data/raw/audio/Husary_128kbps_Mujawwad/001002.md5
- data/raw/audio/Husary_128kbps_Mujawwad/001001.mp3


In [4]:
run_cmd([
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "build.py"),
    "--config",
    str(CONFIG_PATH),
])

Running:
/home/jrilym/Projects/AI/model-asr-quran/.venv/bin/python /home/jrilym/Projects/AI/model-asr-quran/scripts/build.py --config /home/jrilym/Projects/AI/model-asr-quran/configs/local_3050.yaml


Saving the dataset (1/1 shards): 100%|██████████| 47/47 [00:00<00:00, 21565.72 examples/s]
  train: 1914 rows
  validation: 14 rows
  test: 47 rows
[build] saved DatasetDict to data/processed


In [5]:
run_cmd([
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "build_vocab.py"),
    "--config",
    str(CONFIG_PATH),
])

Running:
/home/jrilym/Projects/AI/model-asr-quran/.venv/bin/python /home/jrilym/Projects/AI/model-asr-quran/scripts/build_vocab.py --config /home/jrilym/Projects/AI/model-asr-quran/configs/local_3050.yaml
{
 "َ": 0,
 "ِ": 1,
 "ْ": 2,
 "ُ": 3,
 "ل": 4,
 "ن": 5,
 "م": 6,
 "ا": 7,
 "و": 8,
 "ّ": 9,
 "ي": 10,
 "ر": 11,
 "ٱ": 12,
 "ه": 13,
 "ب": 14,
 "ك": 15,
 "ٰ": 16,
 "ت": 17,
 "ع": 18,
 "ف": 19,
 "أ": 20,
 "ى": 21,
 "ق": 22,
 "س": 23,
 "د": 24,
 "ذ": 25,
 "إ": 26,
 "ح": 27,
 "ٓ": 28,
 "ٍ": 29,
 "ج": 30,
 "ً": 31,
 "ٌ": 32,
 "ة": 33,
 "خ": 34,
 "ص": 35,
 "ء": 36,
 "ش": 37,
 "ز": 38,
 "ط": 39,
 "ث": 40,
 "غ": 41,
 "ض": 42,
 "ئ": 43,
 "ظ": 44,
 "ٔ": 45,
 "ؤ": 46,
 "|": 47,
 "[UNK]": 48,
 "[PAD]": 49
}


[vocab] 50 tokens from dataset(train+validation) (1928 texts) -> data/artifacts/vocab.json


In [6]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
VOCAB_PATH = PROJECT_ROOT / "data" / "artifacts" / "vocab.json"

print("Processed dir:", PROCESSED_DIR, "OK" if PROCESSED_DIR.exists() else "MISSING")
print("Vocab path:", VOCAB_PATH, "OK" if VOCAB_PATH.exists() else "MISSING")

print("\nIsi data/processed:")
if PROCESSED_DIR.exists():
    for path in sorted(PROCESSED_DIR.rglob("*"))[:50]:
        print("-", path.relative_to(PROJECT_ROOT))

Processed dir: /home/jrilym/Projects/AI/model-asr-quran/data/processed OK
Vocab path: /home/jrilym/Projects/AI/model-asr-quran/data/artifacts/vocab.json OK

Isi data/processed:
- data/processed/dataset_dict.json
- data/processed/test
- data/processed/test/cache-9d88e28de32a507d.arrow
- data/processed/test/data-00000-of-00001.arrow
- data/processed/test/dataset_info.json
- data/processed/test/state.json
- data/processed/train
- data/processed/train/cache-4fcd92e88f7864e2.arrow
- data/processed/train/data-00000-of-00001.arrow
- data/processed/train/dataset_info.json
- data/processed/train/state.json
- data/processed/train/tmpbsdo7666
- data/processed/validation
- data/processed/validation/cache-54686eac0c5f9f46.arrow
- data/processed/validation/data-00000-of-00001.arrow
- data/processed/validation/dataset_info.json
- data/processed/validation/state.json


In [7]:
import json
import math
import random
import numpy as np

import torch
import gc

torch.cuda.empty_cache()
gc.collect()

from pathlib import Path
from datasets import load_from_disk, DatasetDict
from transformers import (
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    get_linear_schedule_with_warmup,
)

from torch.utils.data import DataLoader
from tqdm.auto import tqdm

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/home/jrilym/Projects/AI/model-asr-quran/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.12.1+cu130
CUDA available: True
Device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [8]:
import yaml

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg

{'run_name': 'husary_mujawwad_local_3050_small_v1',
 'seed': 42,
 'data': {'reciters': ['Husary_128kbps_Mujawwad'],
  'audio_dir': 'data/raw/audio',
  'text_path': 'data/raw/text/quran_uthmani.json',
  'processed_dir': 'data/processed',
  'sample_rate': 16000,
  'surahs': 'all',
  'max_duration_sec': 20,
  'split': {'strategy': 'by_surah',
   'test_surahs': [78, 93, 110],
   'val_surahs': [94, 109],
   'val_frac_within_train': 0.05}},
 'model': {'base': 'facebook/wav2vec2-xls-r-300m',
  'vocab_path': 'data/artifacts/vocab.json',
  'freeze_feature_encoder_steps': 0,
  'apply_spec_augment': True},
 'training': {'epochs': 30,
  'per_device_train_batch_size': 1,
  'per_device_eval_batch_size': 1,
  'gradient_accumulation_steps': 16,
  'learning_rate': 5e-05,
  'warmup_ratio': 0.1,
  'weight_decay': 0.0,
  'gradient_checkpointing': True,
  'fp16': True,
  'masking_time_prob': 0.05,
  'eval_steps': 500,
  'save_steps': 500,
  'early_stopping_patience': 5},
 'logging': {'hub_repo': None,
  'o

In [9]:
from datasets import load_from_disk, DatasetDict

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Coba format DatasetDict langsung
if (PROCESSED_DIR / "dataset_dict.json").exists():
    dataset = load_from_disk(str(PROCESSED_DIR))

# Coba format split folder: train/validation/test atau train/val/test
elif (PROCESSED_DIR / "train").exists():
    split_map = {}

    for split_name in ["train", "validation", "val", "test"]:
        split_dir = PROCESSED_DIR / split_name
        if split_dir.exists():
            key = "validation" if split_name == "val" else split_name
            split_map[key] = load_from_disk(str(split_dir))

    dataset = DatasetDict(split_map)

else:
    raise RuntimeError(
        f"Format processed dataset belum dikenali di: {PROCESSED_DIR}\n"
        "Cek output scripts/build.py dulu."
    )

dataset

DatasetDict({
    train: Dataset({
        features: ['audio_path', 'text', 'text_plain', 'surah', 'ayah', 'reciter', 'duration'],
        num_rows: 1914
    })
    validation: Dataset({
        features: ['audio_path', 'text', 'text_plain', 'surah', 'ayah', 'reciter', 'duration'],
        num_rows: 14
    })
    test: Dataset({
        features: ['audio_path', 'text', 'text_plain', 'surah', 'ayah', 'reciter', 'duration'],
        num_rows: 47
    })
})

In [10]:
print(dataset)

for split in dataset.keys():
    print("\nSPLIT:", split)
    print("Columns:", dataset[split].column_names)
    print("Rows:", len(dataset[split]))
    print("Sample:")
    print(dataset[split][0])
    break

DatasetDict({
    train: Dataset({
        features: ['audio_path', 'text', 'text_plain', 'surah', 'ayah', 'reciter', 'duration'],
        num_rows: 1914
    })
    validation: Dataset({
        features: ['audio_path', 'text', 'text_plain', 'surah', 'ayah', 'reciter', 'duration'],
        num_rows: 14
    })
    test: Dataset({
        features: ['audio_path', 'text', 'text_plain', 'surah', 'ayah', 'reciter', 'duration'],
        num_rows: 47
    })
})

SPLIT: train
Columns: ['audio_path', 'text', 'text_plain', 'surah', 'ayah', 'reciter', 'duration']
Rows: 1914
Sample:
{'audio_path': 'data/raw/audio/Husary_128kbps_Mujawwad/053021.mp3', 'text': 'أَلَكُمُ ٱلذَّكَرُ وَلَهُ ٱلْأُنثَىٰ', 'text_plain': 'ألكم ٱلذكر وله ٱلأنثى', 'surah': 53, 'ayah': 21, 'reciter': 'Husary_128kbps_Mujawwad', 'duration': 9.691375}


In [11]:
VOCAB_PATH = PROJECT_ROOT / "data" / "artifacts" / "vocab.json"

if not VOCAB_PATH.exists():
    raise FileNotFoundError(f"Vocab belum ada: {VOCAB_PATH}")

tokenizer = Wav2Vec2CTCTokenizer(
    str(VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=cfg["data"]["sample_rate"],
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

print("Processor OK")
print("Vocab size:", len(processor.tokenizer))

Processor OK
Vocab size: 52


In [12]:
sample_split = "train" if "train" in dataset else list(dataset.keys())[0]
columns = dataset[sample_split].column_names

audio_candidates = ["audio", "audio_path", "path", "file", "file_path", "input_path"]
text_candidates = ["text", "sentence", "transcript", "transcription", "target", "label"]

audio_col = next((c for c in audio_candidates if c in columns), None)
text_col = next((c for c in text_candidates if c in columns), None)

print("Columns:", columns)
print("Detected audio_col:", audio_col)
print("Detected text_col:", text_col)

if audio_col is None:
    raise RuntimeError("Kolom audio tidak ketemu. Lihat output columns di atas.")

if text_col is None:
    raise RuntimeError("Kolom text/transcript tidak ketemu. Lihat output columns di atas.")

Columns: ['audio_path', 'text', 'text_plain', 'surah', 'ayah', 'reciter', 'duration']
Detected audio_col: audio_path
Detected text_col: text


In [13]:
import librosa

SAMPLE_RATE = cfg["data"]["sample_rate"]

def prepare_batch(batch):
    audio_value = batch[audio_col]

    # Kalau audio sudah berupa dict dari datasets Audio
    if isinstance(audio_value, dict) and "array" in audio_value:
        speech_array = audio_value["array"]
        sampling_rate = audio_value.get("sampling_rate", SAMPLE_RATE)

        if sampling_rate != SAMPLE_RATE:
            speech_array = librosa.resample(
                speech_array,
                orig_sr=sampling_rate,
                target_sr=SAMPLE_RATE,
            )

    # Kalau audio berupa path string
    else:
        audio_path = Path(audio_value)

        if not audio_path.is_absolute():
            audio_path = PROJECT_ROOT / audio_path

        speech_array, _ = librosa.load(str(audio_path), sr=SAMPLE_RATE)

    inputs = processor(
        speech_array,
        sampling_rate=SAMPLE_RATE,
    )

    batch["input_values"] = inputs.input_values[0]
    batch["attention_mask"] = inputs.attention_mask[0] if "attention_mask" in inputs else None

    labels = processor.tokenizer(batch[text_col]).input_ids
    batch["labels"] = labels

    return batch

In [14]:
remove_columns = dataset[sample_split].column_names

prepared_dataset = dataset.map(
    prepare_batch,
    remove_columns=remove_columns,
    num_proc=1,
)

prepared_dataset

DatasetDict({
    train: Dataset({
        features: ['input_values', 'attention_mask', 'labels'],
        num_rows: 1914
    })
    validation: Dataset({
        features: ['input_values', 'attention_mask', 'labels'],
        num_rows: 14
    })
    test: Dataset({
        features: ['input_values', 'attention_mask', 'labels'],
        num_rows: 47
    })
})

In [15]:
class DataCollatorCTCWithPadding:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        input_features = [
            {"input_values": feature["input_values"]}
            for feature in features
        ]

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=True,
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=True,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100,
        )

        batch["labels"] = labels

        return batch

data_collator = DataCollatorCTCWithPadding(processor)

In [16]:
train_batch_size = cfg["training"]["per_device_train_batch_size"]
eval_batch_size = cfg["training"]["per_device_eval_batch_size"]

train_loader = DataLoader(
    prepared_dataset["train"],
    batch_size=train_batch_size,
    shuffle=True,
    collate_fn=data_collator,
)

eval_split = "validation" if "validation" in prepared_dataset else "val" if "val" in prepared_dataset else "test"

eval_loader = DataLoader(
    prepared_dataset[eval_split],
    batch_size=eval_batch_size,
    shuffle=False,
    collate_fn=data_collator,
)

print("Train batches:", len(train_loader))
print("Eval split:", eval_split)
print("Eval batches:", len(eval_loader))

Train batches: 1914
Eval split: validation
Eval batches: 14


In [17]:
import torch
import gc
from transformers import Wav2Vec2ForCTC

torch.cuda.empty_cache()
gc.collect()

base_model = cfg["model"]["base"]

model = Wav2Vec2ForCTC.from_pretrained(
    base_model,
    vocab_size=len(processor.tokenizer),
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    ignore_mismatched_sizes=True,
    ctc_zero_infinity=True,
)

model.config.ctc_zero_infinity = True
model.config.pad_token_id = processor.tokenizer.pad_token_id

if cfg["model"].get("apply_spec_augment", True):
    model.config.apply_spec_augment = True

# Freeze CNN feature encoder
model.freeze_feature_encoder()

# Freeze semua encoder dulu
freeze_n_layers = len(model.wav2vec2.encoder.layers)

for layer in model.wav2vec2.encoder.layers[:freeze_n_layers]:
    for param in layer.parameters():
        param.requires_grad = False

# Pastikan hanya lm_head yang trainable
for name, param in model.named_parameters():
    if "lm_head" not in name:
        param.requires_grad = False
    else:
        param.requires_grad = True

model.to(device)

trainable_parameters = [p for p in model.parameters() if p.requires_grad]

trainable_params = sum(p.numel() for p in trainable_parameters)
total_params = sum(p.numel() for p in model.parameters())

print("Model loaded:", base_model)
print("Frozen encoder layers:", freeze_n_layers)
print("Trainable params:", trainable_params)
print("Total params:", total_params)
print("Trainable ratio:", round(trainable_params / total_params * 100, 4), "%")

if torch.cuda.is_available():
    print("Allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 2))
    print("Reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 2))

Loading weights: 100%|██████████| 422/422 [00:00<00:00, 66386.48it/s]
[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
project_hid.weight           | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: facebook/wav2vec2-xls-r-300m
Frozen encoder layers: 24
Trainable params: 53300
Total params: 315492020
Trainable ratio: 0.0169 %
Allocated GB: 1.18
Reserved GB: 1.19


In [18]:
epochs = cfg["training"]["epochs"]
learning_rate = float(cfg["training"]["learning_rate"])
warmup_ratio = float(cfg["training"].get("warmup_ratio", 0.1))
weight_decay = float(cfg["training"].get("weight_decay", 0.0))
grad_accum = int(cfg["training"].get("gradient_accumulation_steps", 1))

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=learning_rate,
    weight_decay=weight_decay,
    foreach=False,
)

num_update_steps_per_epoch = math.ceil(len(train_loader) / grad_accum)
max_train_steps = epochs * num_update_steps_per_epoch
num_warmup_steps = int(max_train_steps * warmup_ratio)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=max_train_steps,
)

print("Epochs:", epochs)
print("Gradient accumulation:", grad_accum)
print("Max train steps:", max_train_steps)
print("Warmup steps:", num_warmup_steps)
print("Trainable params in optimizer:", sum(p.numel() for p in trainable_parameters))

Epochs: 30
Gradient accumulation: 16
Max train steps: 3600
Warmup steps: 360
Trainable params in optimizer: 53300


In [19]:
OUTPUT_DIR = PROJECT_ROOT / cfg["logging"]["output_dir"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)

OUTPUT_DIR: /home/jrilym/Projects/AI/model-asr-quran/data/artifacts/checkpoints/local_3050_small


In [20]:
import json
from pathlib import Path

def save_training_state(path, epoch, global_step, best_eval_loss, history):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)

    # Save model + processor
    model.save_pretrained(path)
    processor.save_pretrained(path)

    # Save optimizer/scheduler/training state
    torch.save(
        {
            "epoch": epoch,
            "global_step": global_step,
            "best_eval_loss": best_eval_loss,
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
        },
        path / "training_state.pt",
    )

    # Save history loss
    with open(path / "training_history.json", "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

    print(f"Saved training state to: {path}")


def load_training_history(path):
    history_path = Path(path) / "training_history.json"

    if history_path.exists():
        with open(history_path, "r", encoding="utf-8") as f:
            return json.load(f)

    return {
        "epoch": [],
        "train_loss": [],
        "eval_loss": [],
    }

In [21]:
RESUME = True
RESUME_DIR = OUTPUT_DIR / "latest"

start_epoch = 0
global_step = 0
best_eval_loss = float("inf")

history = {
    "epoch": [],
    "train_loss": [],
    "eval_loss": [],
}

state_path = RESUME_DIR / "training_state.pt"

if RESUME and state_path.exists():
    print("Resuming from:", RESUME_DIR)

    model = Wav2Vec2ForCTC.from_pretrained(RESUME_DIR)
    model.to(device)

    for name, param in model.named_parameters():
        param.requires_grad = False

    for param in model.lm_head.parameters():
        param.requires_grad = True

    trainable_parameters = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(
        trainable_parameters,
        lr=learning_rate,
        weight_decay=weight_decay,
        foreach=False,
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=max_train_steps,
    )

    state = torch.load(state_path, map_location=device)

    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])

    start_epoch = state["epoch"] + 1
    global_step = state["global_step"]
    best_eval_loss = state["best_eval_loss"]

    history = load_training_history(RESUME_DIR)

    print("Resume OK")
    print("Start epoch:", start_epoch + 1)
    print("Global step:", global_step)
    print("Best eval loss:", best_eval_loss)
    print("History length:", len(history["epoch"]))

else:
    print("No resume checkpoint found. Starting fresh.")

Resuming from: /home/jrilym/Projects/AI/model-asr-quran/data/artifacts/checkpoints/local_3050_small/latest


Loading weights: 100%|██████████| 424/424 [00:00<00:00, 1285.58it/s]


Resume OK
Start epoch: 6
Global step: 916
Best eval loss: 7.8383912699563165
History length: 7


In [ ]:
from torch.nn.utils import clip_grad_norm_

logging_steps = int(cfg["training"].get("logging_steps", 25))
save_steps = int(cfg["training"].get("save_steps", 500))

for epoch in range(start_epoch, epochs):
    model.train()
    total_train_loss = 0.0
    valid_steps = 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}")

    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(progress):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss / grad_accum

        if not torch.isfinite(loss):
            print(f"Non-finite loss detected at batch {step}: {loss.item()}")
            optimizer.zero_grad(set_to_none=True)
            continue

        loss.backward()

        total_train_loss += loss.item() * grad_accum
        valid_steps += 1

        if (step + 1) % grad_accum == 0 or (step + 1) == len(train_loader):
            clip_grad_norm_(trainable_parameters, 0.5)

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1

            if global_step % logging_steps == 0:
                avg_loss = total_train_loss / max(valid_steps, 1)
                progress.set_postfix({
                    "loss": round(avg_loss, 4),
                    "step": global_step,
                })

            if global_step % save_steps == 0:
                ckpt_dir = OUTPUT_DIR / f"checkpoint-{global_step}"

                save_training_state(
                    ckpt_dir,
                    epoch=epoch,
                    global_step=global_step,
                    best_eval_loss=best_eval_loss,
                    history=history,
                )

    avg_train_loss = total_train_loss / max(valid_steps, 1)
    print(f"\nEpoch {epoch + 1} train loss: {avg_train_loss:.4f}")

    model.eval()
    total_eval_loss = 0.0
    valid_eval_steps = 0

    with torch.no_grad():
        for batch in tqdm(eval_loader, desc="Evaluating"):
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            eval_loss = outputs.loss

            if not torch.isfinite(eval_loss):
                continue

            total_eval_loss += eval_loss.item()
            valid_eval_steps += 1
    
    avg_eval_loss = total_eval_loss / max(valid_eval_steps, 1)
    print(f"Epoch {epoch + 1} eval loss: {avg_eval_loss:.4f}")

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(avg_train_loss)
    history["eval_loss"].append(avg_eval_loss)

    if avg_eval_loss < best_eval_loss:
        best_eval_loss = avg_eval_loss

        best_dir = OUTPUT_DIR / "best"
        save_training_state(
            best_dir,
            epoch=epoch,
            global_step=global_step,
            best_eval_loss=best_eval_loss,
            history=history,
        )

        print(f"New best checkpoint saved: {best_dir}")

    latest_dir = OUTPUT_DIR / "latest"
    save_training_state(
        latest_dir,
        epoch=epoch,
        global_step=global_step,
        best_eval_loss=best_eval_loss,
        history=history,
    )
    print(f"Latest checkpoint saved: {latest_dir}")

final_dir = OUTPUT_DIR / "final"
final_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(final_dir)
processor.save_pretrained(final_dir)

with open(final_dir / "training_history.json", "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)

print("Training done.")
print("Final checkpoint:", final_dir)
print("Best eval loss:", best_eval_loss)

Epoch 6/30:  70%|███████   | 1344/1914 [12:59<08:47,  1.08it/s, loss=21.5, step=1000]

Saved training state to: /home/jrilym/Projects/AI/model-asr-quran/data/artifacts/checkpoints/local_3050_small/checkpoint-1000


Epoch 6/30: 100%|██████████| 1914/1914 [18:28<00:00,  1.73it/s, loss=21.7, step=1025]



Epoch 6 train loss: 21.8622


Evaluating: 100%|██████████| 14/14 [00:02<00:00,  4.70it/s]


...

Evaluating: 100%|██████████| 14/14 [00:02<00:00,  4.87it/s]


Epoch 12 eval loss: 8.5671


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.84s/it]


Saved training state to: /home/jrilym/Projects/AI/model-asr-quran/data/artifacts/checkpoints/local_3050_small/latest
Latest checkpoint saved: /home/jrilym/Projects/AI/model-asr-quran/data/artifacts/checkpoints/local_3050_small/latest


Epoch 13/30:  58%|█████▊    | 1103/1914 [10:48<07:17,  1.85it/s, loss=16.2, step=1800]

In [ ]:
model.eval()

sample = prepared_dataset[eval_split][0]

inputs = processor.pad(
    [{"input_values": sample["input_values"]}],
    padding=True,
    return_tensors="pt",
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    logits = model(**inputs).logits

pred_ids = torch.argmax(logits, dim=-1)
pred_text = processor.batch_decode(pred_ids)[0]

label_ids = np.array(sample["labels"])
label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
target_text = processor.decode(label_ids)

print("PRED:")
print(pred_text)

print("\nTARGET:")
print(target_text)

PRED:
ى أ أ أ أ أ أ أ أ أ ٔأ أ أ أ أ أ أ أ أ أ أ ٔ ٔ أ ٔ ٔ  ٔ  ٔ ٔ ٔ أ ٔ ٔ ٔ ٔ ٔ أ أ أ أ ى

TARGET:
أَلَمْ نَشْرَحْ لَكَ صَدْرَكَ


In [ ]:
import matplotlib.pyplot as plt

if len(history["epoch"]) == 0:
    print("History masih kosong. Jalankan training minimal 1 epoch dulu.")
else:
    plt.figure(figsize=(8, 5))

    plt.plot(history["epoch"], history["train_loss"], marker="o", label="Train loss")
    plt.plot(history["epoch"], history["eval_loss"], marker="o", label="Eval loss")

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs Evaluation Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
plot_path = OUTPUT_DIR / "training_loss_curve.png"

if len(history["epoch"]) > 0:
    plt.figure(figsize=(8, 5))

    plt.plot(history["epoch"], history["train_loss"], marker="o", label="Train loss")
    plt.plot(history["epoch"], history["eval_loss"], marker="o", label="Eval loss")

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs Evaluation Loss")
    plt.legend()
    plt.grid(True)

    plt.savefig(plot_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved plot to:", plot_path)
else:
    print("History masih kosong.")

In [ ]:
import json

metrics_path = OUTPUT_DIR / "training_history.json"

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)

print("Training history saved to:", metrics_path)

In [ ]:
import pandas as pd

df_history = pd.DataFrame(history)
csv_path = OUTPUT_DIR / "training_history.csv"

df_history.to_csv(csv_path, index=False)

print("CSV saved to:", csv_path)
df_history